# Decoding device detection events with PyMatching

SwiftQuantum Core runs QEC circuits and records detection events. **It does not decode.** This notebook shows the decoding happening outside the package, in Stim and PyMatching.

Circuit: QECirc #1289, rotated surface code distance 3, AlphaSyndrome schedule (CC BY-SA 4.0, see `third_party/qecirc/README.md`), with the package's uniform noise model at p = 0.005.

Produce the inputs from the repository root first:

```bash
swift run -c release qecirc-surface-memory 0.005 20000
pip install stim pymatching numpy
```

That writes `qecirc_d3_memory.stim` (the noisy circuit) and `qecirc_d3_memory.01` (one shot per line: 24 detectors, then the observable).

In [1]:
import numpy as np
import pymatching
import stim

circuit = stim.Circuit(open("../qecirc_d3_memory.stim").read())
rows = [line.strip() for line in open("../qecirc_d3_memory.01") if line.strip()]
events = np.array([[c == "1" for c in row] for row in rows], dtype=bool)
n_det = circuit.num_detectors
device_det, device_obs = events[:, :n_det], events[:, n_det:]
print(f"stim {stim.__version__}, pymatching {pymatching.__version__}")
print(f"{len(rows)} shots, {n_det} detectors, {circuit.num_observables} observable")

stim 1.16.0, pymatching 2.4.0
20000 shots, 24 detectors, 1 observable


## 1. The same circuit sampled by Stim, the reference stabilizer simulator

In [2]:
stim_det, stim_obs = circuit.compile_detector_sampler(seed=1).sample(len(rows), separate_observables=True)
print(f"detection fraction    device {device_det.mean():.4f}   stim {stim_det.mean():.4f}")
print(f"raw observable flips  device {device_obs.mean():.4f}   stim {stim_obs.mean():.4f}")

detection fraction    device 0.0694   stim 0.0696
raw observable flips  device 0.1253   stim 0.1192


## 2. Decode both sets of detection events with minimum-weight perfect matching

In [3]:
dem = circuit.detector_error_model(decompose_errors=True)
matching = pymatching.Matching.from_detector_error_model(dem)
for name, det, obs in (("device", device_det, device_obs), ("stim", stim_det, stim_obs)):
    predicted = matching.decode_batch(det)
    print(f"{name:6s} logical error rate after decoding: {np.any(predicted != obs, axis=1).mean():.4f}"
          f"   (before decoding: {obs.mean():.4f})")

device logical error rate after decoding: 0.0216   (before decoding: 0.1253)
stim   logical error rate after decoding: 0.0208   (before decoding: 0.1192)


Decoding lowers the logical error rate by roughly a factor of six on both sets of events. The device's events decode to the same rate as Stim's, within sampling noise. Pseudo-random sampling on the device, a simulator and not hardware.